# NIS 2019 — cohort extraction by diagnosis code

Pull every discharge carrying a given ICD-10-CM code in **any** of the 40
diagnosis slots, then describe that cohort.

Data: HCUP National Inpatient Sample 2019 Core, 7,083,805 discharges.
Restricted under a Data Use Agreement — see `DATA_HANDLING.md` before you
push anything anywhere.

Run `python src/build_transactions.py` once before this notebook; it turns
the 2 GB `.SAV` into a sparse matrix in `cache/`.

Everything reported here is aggregate and passes the HCUP cell-size rule —
no statistic may rest on 10 or fewer discharges.

In [ ]:
import sys
sys.path.insert(0, "src")

import numpy as np
import pandas as pd

from dataset import Dataset
from config import MIN_CELL

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 70)

nis = Dataset()
print(f"{len(nis):,} discharges | {len(nis.codes):,} distinct ICD-10-CM codes")

## Pick a code

`P15.9` — birth injury, unspecified — as an example. Change `CODE` to look at
anything else; nothing below is hard-coded to it.

Codes are matched exactly, so check the one you want exists first: `P159` is
valid, `P1590` is not, and a typo just returns an empty cohort rather than an
error.

In [ ]:
CODE = "P159"

print(f"{CODE}: {nis.describe(CODE)}")
mask = nis.has_code(CODE)
print(f"{mask.sum():,} discharges carry it in at least one diagnosis slot")

### One mask, not forty filters

A discharge can carry the same code in more than one of the 40 slots. Building
one filter per slot and stacking the results with `concat` would count that
discharge twice; a single column of the discharge × code matrix is already the
union across all 40 positions, so `has_code` cannot double-count.

## Cohort profile\n\nAggregate only — no record-level values leave this cell.

In [ ]:
profile = nis.profile(mask)
profile.round(2).to_frame(f"{CODE} cohort")

In [ ]:
everyone = nis.profile(np.ones(len(nis), dtype=bool))
comparison = pd.concat(
    [nis.profile(mask).rename(CODE), everyone.rename("all discharges")], axis=1
)
comparison["ratio"] = comparison[CODE] / comparison["all discharges"]
comparison.round(2)

## What else is coded on these discharges

`lift` = how much more often a code appears in this cohort than in the NIS
overall. Rows resting on 10 or fewer discharges are dropped before you see
them, per the HCUP cell-size rule.

In [ ]:
co = nis.cooccurring(CODE, min_count=MIN_CELL)
print(f"{len(co):,} codes co-occur at n >= {MIN_CELL}")

co.head(25)[["code", "description", "n_in_cohort", "pct_of_cohort", "lift"]]

Sorted by lift instead — the codes that are *distinctively* associated
rather than merely common. Restricted to codes seen on at least 1% of the
cohort so a handful of discharges cannot manufacture a huge ratio.

In [ ]:
distinctive = co[co["pct_of_cohort"] >= 0.01].nlargest(25, "lift")
distinctive[["code", "description", "n_in_cohort", "pct_of_cohort", "lift"]]

## Record-level export

Only if you genuinely need it, and only into `cache/`, which is gitignored.
Never to the Desktop, Downloads, or any synced folder — that is outside the
project and outside every ignore rule.

Left commented out on purpose.

In [ ]:
# import pyreadstat
# from config import SAV, DX_COLS, CACHE
#
# rows = np.flatnonzero(mask)
# df, _ = pyreadstat.read_sav(str(SAV), usecols=DX_COLS + ["AGE", "FEMALE", "LOS"])
# df.iloc[rows].to_parquet(CACHE / f"cohort_{CODE}.parquet")   # gitignored
#
# Record-level output stays in cache/ and never leaves this machine.